In [2]:
from pipeline import condition_signal, correct_motion, plot_motion_output, sort_ks4, save_binary_recording, run_qc, KilosortResults, load_qc, run_cur, load_cur
from spikeinterface.sorters import get_default_sorter_params
from pathlib import Path
import shutil
# I'm using a pinned version of spikeinterface, so if something doesn't work with the latest version, ask about it
import spikeinterface.full as si

[INFO] - Loading faiss with AVX512 support.
[INFO] - Successfully loaded faiss with AVX512 support.
[INFO] - Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes.


In [3]:
import spikeinterface.extractors as se
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
%matplotlib widget

In [180]:
pipeline_dir=Path('/media/huklab/Data/RyanSorting/pipeline_results_Rocky20240704_V1V2_g0_imec0_remove_end/')
ks4_results = KilosortResults(pipeline_dir / 'kilosort4')
seg_motion =  si.load_extractor(pipeline_dir / 'preprocessed_recording')
ks4_sorter = si.load_extractor(pipeline_dir / 'kilosort4/sorter')

/tmp/ipykernel_2850584/241351080.py:3: DeprecationWarning: load_extractor() is deprecated and will be removed in the future. Please use load() instead.
  seg_motion =  si.load_extractor(pipeline_dir / 'preprocessed_recording')
/tmp/ipykernel_2850584/241351080.py:4: DeprecationWarning: load_extractor() is deprecated and will be removed in the future. Please use load() instead.
  ks4_sorter = si.load_extractor(pipeline_dir / 'kilosort4/sorter')


In [436]:
seg_motion.get_property('gain_to_uV')[0]==2.34375



True

In [156]:
import numpy as np
spike_positions_file = pipeline_dir / 'kilosort4/sorter_output/spike_positions.npy'
spike_positions = np.load(spike_positions_file)
print(spike_positions[:,1])

[3770.373   940.0052 1282.192  ... 1533.7104 1038.147  2824.0637]


In [186]:
kept_spikes=np.load(ks4_results.kept_spikes_file)
sp_t=np.load(ks4_results.spike_times_file)
np.sum(~kept_spikes)/len(kept_spikes)
len(sp_t)/len(kept_spikes)

0.9866266348430104

In [94]:
sp_z= spike_positions[:,1]
sp_t=ks4_sorter.spikes['sample_index']
clu=ks4_results.spike_clusters
thr_z=10
thr_t=1#0.0001*30000 # .1ms
delta_sp_t=np.diff(sp_t)
delta_sp_z=np.diff(sp_z)
delta_clu=np.diff(clu)!=0
#np.sum((delta_sp_t<thr_t)/len(sp_t))
boolll=((delta_sp_t<thr_t)&(delta_sp_z<thr_z)&delta_clu)
duped_spikes=np.nonzero(boolll)
print(100*len(duped_spikes[0])/len(sp_t),"%  duped spikes")
#distance=ks4_sorter.get_unit_property(1,key='amplitude')

6.53219681389056 %  duped spikes


In [154]:
print(ks4_sorter.spikes['sample_index'])
print(ks4_sorter.spikes['unit_index'])
print(len(ks4_sorter.spikes))

[       20        28        29 ... 329999969 329999969 329999986]
[1880  522  790 ...  939  629 1509]
29295290


In [142]:
ks4_sorter

NumpyFolder (NumpyFolderSorting): 1909 units - 1 segments - 30.0kHz

In [150]:
#len(ks4_results.spike_times)
#len(ks4_sorter.spikes)
cache_dir = pipeline_dir / 'kilosort4'
cleaned_sorter=ks4_sorter #Does this actually make a copy? Nope

# ks4_sorter.copy_metadata(cleaned_sorter, only_main=False)
# cleaned_sorter._parent = ks4_sorter
# if ks4_sorter.has_recording():
#     cleaned_sorter.register_recording(ks4_sorter._recording)


cleaned_sorter.spikes=np.delete(cleaned_sorter.spikes,duped_spikes)
print(len(cleaned_sorter.spikes), "remaining of ", len(ks4_sorter.spikes), "total spikes")
cleaned_sorter.save_to_folder(pipeline_dir / 'cur/kilosort4/sorter_output')


27381664 remaining of  27381664 total spikes
Use cache_folder=/media/huklab/Data/RyanSorting/pipeline_results_Rocky20240704_V1V2_g0_imec0_remove_end/cur/kilosort4/sorter_output


AssertionError: folder /media/huklab/Data/RyanSorting/pipeline_results_Rocky20240704_V1V2_g0_imec0_remove_end/cur/kilosort4/sorter_output already exists, choose another name or use overwrite=True

In [151]:
cleaned_sorter.save_to_folder(pipeline_dir / 'cur/kilosort4/sorter_output',overwrite=True)


Use cache_folder=/media/huklab/Data/RyanSorting/pipeline_results_Rocky20240704_V1V2_g0_imec0_remove_end/cur/kilosort4/sorter_output


NumpyFolder (NumpyFolderSorting): 1909 units - 1 segments - 30.0kHz

In [ ]:
from kilosort import io
results_dir, similar_templates, is_ref, est_contam_rate, kept_spikes = \
    io.save_to_phy(
        st, clu, tF, Wall, ops['probe'], ops, imin, results_dir=results_dir,
        data_dtype=ops['data_dtype'], save_extra_vars=save_extra_vars,
        save_preprocessed_copy=save_preprocessed_copy
        )

In [126]:
print(len(ks4_results.spike_templates))

29295290


In [74]:
thr_t

3.0

In [188]:
from spikeinterface.extractors import read_phy

# For kilosort/phy output files we can use the read_phy
# most formats will have a read_xx that can used.
phy_sorting = read_phy('/media/huklab/Data/RyanSorting/pipeline_results_Rocky20240704_V1V2_g0_imec0_remove_end/kilosort4/sorter_output/')
phy_sorting

PhySortingExtractor: 1909 units - 1 segments - 30.0kHz

In [ ]:
cur_results = run_cur(seg_motion, ks4_sorter, pipeline_dir / 'cur', recalc=True)

In [239]:
import os
import re
phypath=pipeline_dir / 'kilosort4/sorter_output/'
filelist=os.listdir(phypath)
npyfile=[False]*len(filelist)
tsvfile=[False]*len(filelist)

for ii in range(len(filelist)):
    file=filelist[ii]
    #print(file)
    if bool(re.search(".npy",file)):
        npyfile[ii]=True
    if bool(re.search(".tsv",file)):
        tsvfile[ii]=True

npylist = [item for item, select in zip(filelist, npyfile) if select]
tsvlist = [item for item, select in zip(filelist, tsvfile) if select]

print(npylist)
#print(tsvlist)


['pc_feature_ind.npy', 'spike_detection_templates.npy', 'whitening_mat_dat.npy', 'whitening_mat_inv.npy', 'tF.npy', 'spike_templates.npy', 'full_amp.npy', 'templates.npy', 'pc_features.npy', 'channel_shanks.npy', 'full_st.npy', 'channel_map.npy', 'amplitudes.npy', 'spike_clusters.npy', 'kept_spikes.npy', 'ops.npy', 'channel_positions.npy', 'spike_positions.npy', 'Wall.npy', 'spike_times.npy', 'whitening_mat.npy', 'templates_ind.npy', 'similar_templates.npy', 'full_clu.npy']


In [283]:
pipeline_dir.parent

PosixPath('/media/huklab/Data/RyanSorting')

In [280]:
n_spikes0=len(sp_t)
n_clu0=len(ks4_results.cluster_labels)

filecheck1=np.load(phypath / npylist[0])
result=([dim==n_clu0 for dim in filecheck1.shape])
clu_dim=np.argwhere(result)
print(clu_dim[0][0])

print(any(result))

0
True


In [306]:
n_spikes0=len(sp_t)
n_clu0=len(ks4_results.cluster_labels)


for ii in range(len(npylist)):
    filecheck=np.load(phypath / npylist[ii],allow_pickle=True)
    print(npylist[ii], filecheck.shape)
    spfind=([dim==n_spikes0 for dim in filecheck.shape])
    sp_dim=np.argwhere(spfind)


    #if any(spfind):
        #print("remove duped spikes in spike dimension first")
        #print(sp_dim[0][0])

    clufind=([dim==n_clu0 for dim in filecheck.shape])
    clu_dim=np.argwhere(clufind)
    
    #if any(clufind):
        #print("then merge clusters from remaining spikes")
        #print(clu_dim[0][0])


pc_feature_ind.npy (1909, 10)
spike_detection_templates.npy (29295290,)
whitening_mat_dat.npy (384, 384)
whitening_mat_inv.npy (384, 384)
tF.npy (29692377, 10, 6)
spike_templates.npy (29295290,)
full_amp.npy (29692377,)
templates.npy (1909, 61, 384)
pc_features.npy (29295290, 6, 10)
channel_shanks.npy (384,)
full_st.npy (29692377, 3)
channel_map.npy (384,)
amplitudes.npy (29295290,)
spike_clusters.npy (29295290,)
kept_spikes.npy (29692377,)
ops.npy ()
channel_positions.npy (384, 2)
spike_positions.npy (29295290, 2)
Wall.npy (1909, 384, 6)
spike_times.npy (29295290,)
whitening_mat.npy (384, 384)
templates_ind.npy (1909, 384)
similar_templates.npy (1909, 1909)
full_clu.npy (29692377,)


In [429]:
newphypath=Path('/home/huklab/Documents/RyanSorting/SpikeSortingTools/pipeline_results_Rocky20240704_V1V2_g0_imec0_785s_slice/cur/cur_sorter_output0')

#phypath=pipeline_dir / 'kilosort4/sorter_output/'
filelist=os.listdir(newphypath)
npyfile=[False]*len(filelist)
tsvfile=[False]*len(filelist)

for ii in range(len(filelist)):
    file=filelist[ii]
    #print(file)
    if bool(re.search(".npy",file)):
        npyfile[ii]=True
    if bool(re.search(".tsv",file)):
        tsvfile[ii]=True

npylist = [item for item, select in zip(filelist, npyfile) if select]
tsvlist = [item for item, select in zip(filelist, tsvfile) if select]

print(npylist)
#print(tsvlist)

for ii in range(len(npylist)):
    filecheck=np.load(newphypath / npylist[ii],allow_pickle=True)
    print(npylist[ii], filecheck.shape)
    spfind=([dim==n_spikes0 for dim in filecheck.shape])
    sp_dim=np.argwhere(spfind)


    #if any(spfind):
        #print("remove duped spikes in spike dimension first")
        #print(sp_dim[0][0])

    clufind=([dim==n_clu0 for dim in filecheck.shape])
    clu_dim=np.argwhere(clufind)
    
    #if any(clufind):
        #print("then merge clusters from remaining spikes")
        #print(clu_dim[0][0])


['kept_spikes.npy', 'templates.npy', 'templates_ind.npy', 'channel_map.npy', 'channel_shanks.npy', 'whitening_mat.npy', 'amplitudes.npy', 'similar_templates.npy', 'spike_positions.npy', 'spike_clusters.npy', 'channel_positions.npy', 'whitening_mat_inv.npy', 'spike_detection_templates.npy', 'spike_templates.npy', 'whitening_mat_dat.npy', 'spike_times.npy']
kept_spikes.npy (2002165,)
templates.npy (551, 61, 384)
templates_ind.npy (551, 384)
channel_map.npy (384,)
channel_shanks.npy (384,)
whitening_mat.npy (384, 384)
amplitudes.npy (2002165,)
similar_templates.npy (551, 551)
spike_positions.npy (2002165, 2)
spike_clusters.npy (2002165,)
channel_positions.npy (384, 2)
whitening_mat_inv.npy (384, 384)
spike_detection_templates.npy (2002165,)
spike_templates.npy (2002165,)
whitening_mat_dat.npy (384, 384)
spike_times.npy (2002165,)


In [428]:
datafile=np.load(newphypath / "templates_ind.npy")

In [423]:
filecheck=np.load(newphypath / npylist[ii],allow_pickle=True)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [303]:
npy_path = '/media/huklab/Data/RyanSorting/pipeline_results_Rocky20240704_V1V2_g0_imec0_remove_end/cur/cur_todo_phy.npy'
curation_todo_wrapped = np.load(npy_path, allow_pickle=True)
curation_todo=curation_todo_wrapped.item()
merge_unit_groups=curation_todo['merge_unit_groups']

print(len(merge_unit_groups))

46


In [419]:
tf0=np.load(phypath / 'tF.npy')
kept0=np.load(phypath / 'kept_spikes.npy')
kept=np.argwhere(kept0)
np.shape(tf0[kept])


(29295290, 1, 10, 6)

In [ ]:
np.shape(ks4_results.st)

(29295290,)

In [308]:
pipeline_dir=cache_dir.parent
print((pipeline_dir))

/media/huklab/Data/RyanSorting/pipeline_results_Rocky20240704_V1V2_g0_imec0_remove_end


In [7]:
pipeline_dir= Path('/home/huklab/Documents/RyanSorting/SpikeSortingTools/pipeline_results_Rocky20240704_V1V2_g0_imec0_locarfirst/')
import numpy as np

In [8]:
oldphypath = pipeline_dir / 'kilosort4/sorter_output/'
#newphypath = cache_dir / 'cur_sorter_output/'

ops0_wrapped=np.load(oldphypath / 'ops.npy',allow_pickle=True)
ops0=ops0_wrapped.item()
st0=np.load(oldphypath / 'spike_times.npy')
clu0=np.load(oldphypath / 'spike_clusters.npy')
tF0=np.load(oldphypath / 'tF.npy')
Wall0=np.load(oldphypath / 'Wall.npy')

ops1=ops0
st1=np.delete(st0, duped_spikes, axis=0)
clu1=np.delete(clu0, duped_spikes, axis=0)
tF1=np.delete(tF0, duped_spikes, axis=0)

n_clu0=len(set(clu0))

n_groups=len(merge_unit_groups) # number of groups to merge 
newids=np.max(clu0)+range(n_groups)+1 #append new ids

Wall1=Wall0

NameError: name 'duped_spikes' is not defined

In [12]:
print(ops0)

{'n_chan_bin': 384, 'fs': 30000.074166666665, 'batch_size': 60000, 'nblocks': 0, 'Th_universal': 12, 'Th_learned': 10, 'tmin': 0, 'tmax': inf, 'nt': 61, 'shift': None, 'scale': None, 'artifact_threshold': inf, 'nskip': 25, 'whitening_range': 32, 'highpass_cutoff': 300, 'binning_depth': 5, 'sig_interp': 20, 'drift_smoothing': [0.5, 0.5, 0.5], 'nt0min': 20, 'dmin': 20.0, 'dminx': 32, 'min_template_size': 10, 'template_sizes': 5, 'nearest_chans': 10, 'nearest_templates': 100, 'max_channel_distance': 32, 'max_peels': 100, 'templates_from_data': True, 'n_templates': 6, 'n_pcs': 6, 'Th_single_ch': 6, 'acg_threshold': 0.2, 'ccg_threshold': 0.4, 'cluster_downsampling': 20, 'x_centers': None, 'duplicate_spike_ms': 0.5, 'position_limit': 100, 'filename': '/home/huklab/Documents/RyanSorting/SpikeSortingTools/pipeline_results_Rocky20240704_V1V2_g0_imec0_locarfirst/preprocessed_recording/traces_cached_seg0.raw', 'data_dir': '/home/huklab/Documents/RyanSorting/SpikeSortingTools/pipeline_results_Rock

In [340]:
Wall1=Wall0
nchan=np.size(Wall0,axis=1)
ntp=np.size(Wall0,axis=2)
for ii in range(n_groups):
    n_clu=len(merge_unit_groups[ii])
    nspikes=[0]*(n_clu)
    #nspikes[jj]=np.sum(clu1==merge_unit_groups[ii][jj])
    for jj in range(n_clu):
        nspikes[jj]=np.sum(clu1==merge_unit_groups[ii][jj]) #count to decide which waveform to keep
    best_unit_clu=merge_unit_groups[ii][np.argmax(nspikes)]
    best_unit_idx=np.argwhere(ks4_sorter.unit_ids==best_unit_clu)
    #Wall1[n_clu0+ii]=Wall0[best_unit_idx[0][0],:,:] #copy waveforms into the next slot
    appendthis=np.reshape(Wall0[best_unit_idx[0][0],:,:],newshape=[1,nchan,ntp])
    np.append(Wall1,appendthis,0)

In [350]:
Wall_remove_idx=[]
ii=0
n_clu=len(merge_unit_groups[ii])
jj=0
cluster_change_idx=np.argwhere(ks4_sorter.unit_ids==merge_unit_groups[ii][jj])
Wall_remove_idx=np.append(Wall_remove_idx,cluster_change_idx)
#cluster_change_idx=clu1==merge_unit_groups[ii][jj]


In [351]:
print(Wall_remove_idx)

[0.]


In [389]:
Wall_remove_idx=[]
for ii in range(n_groups):
    n_clu=len(merge_unit_groups[ii])
    for jj in range(n_clu):
        clu1[np.argwhere(clu1==merge_unit_groups[ii][jj])]=newids[ii]
        #Remove entries in Wall, dim 0
        cluster_change_idx=np.argwhere(ks4_sorter.unit_ids==merge_unit_groups[ii][jj])
        Wall_remove_idx=np.append(Wall_remove_idx,cluster_change_idx)

print('removing', len(set(Wall_remove_idx)),' waveforms')
Wall1=np.delete(Wall0,Wall_remove_idx.astype(int),axis=0)

removing 469  waveforms


In [388]:
Wall_remove_idx.astype(int)

array([   0,    7,   23,   40,   55,   99,  118,  124,  127,  159,  168,
        170,  172,  205,  390,  403,  460,  468,  555,  560,  562,  563,
        636,  645,  695,  744,  746,  755,  756,  758,  760,  761,  762,
        766,  770,  773,  774,  776,  778,  783,  794,  795,  801,  805,
        806,  809,  810,  812,  814,  815,  818,  827,  840,  844,  861,
        869,  870,  876,  877,  881,  887,  894,  896,  898,  900,  901,
        902,  904,  906,  909,  923,  924,  927,  937,  938,  954,  955,
        958,  961,  965, 1002, 1036, 1038, 1040, 1041, 1043, 1044, 1046,
       1055, 1077, 1095, 1101, 1110, 1113, 1125, 1130, 1171, 1178, 1193,
       1218, 1227, 1234, 1244, 1248, 1253, 1259, 1261, 1262, 1274, 1292,
       1294, 1296, 1329, 1342, 1372, 1385, 1487, 1489, 1491, 1525, 1561,
       1562, 1586, 1597, 1600, 1651, 1693, 1712, 1736, 1758, 1784, 1820,
       1823, 1849, 1853, 1895, 1901,    3,   25,   45,   50,   54,   72,
         81,   83,   95,   97,   98,  101,  108,  1

In [393]:
ks4_results_clean=ks4_results

In [ ]:
# ks4_results_clean.cluster_labels
ks4_results_clean.spike_amplitudes
ks4_results_clean.cluster_labels
ks4_results_clean.spike_clusters
ks4_results_clean.spike_positions
ks4_results_clean.spike_templates
ks4_results_clean.spike_times
ks4_results_clean.st

AttributeError: 'KilosortResults' object has no attribute 'spike_positions'

In [409]:
type(ks4_results_clean.cluster_labels)

pandas.core.frame.DataFrame